# Week 7 Assignment - Delta Lake

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

## Loading Customer Master Dataset

In [0]:
master_df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/my_catalog/demo/all_volumes/Sample - Superstore (1).csv")
master_df.show(10);

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
## Checking Schema

In [0]:
master_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



## Checking for Null Values

In [0]:
master_df.select([sum(col(c).isNull().cast("int")).alias(c)
for c in master_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [0]:
## Removing Duplicate Records

In [0]:
master_df = master_df.dropDuplicates()
master_df.count()

9994

## Rename Columns

In [0]:
from pyspark.sql.functions import col
df = master_df.select([
    col(c).alias(c.replace(" ", "_"))
    for c in master_df.columns
])

df.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)




##Create Delta Table

In [0]:
delta_path = "/Volumes/workspace/default/my_files/superstore_delta"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [0]:
delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.toDF().show(10)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|       State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name| Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|     5|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|   Consumer|United States|Fort Lauderdale|     Florida|      33311|  South|OFF-ST-10000760|Office Supplies| 

## 5. Data Cleaning

In [0]:
df = delta_table.toDF()

df = df.dropDuplicates()


In [0]:
df = df.fillna(0)

In [0]:
delta_path="/Volumes/workspace/default/my_files/superstore_delta";
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/my_files/superstore_delta")

delta_table = DeltaTable.forPath(spark, delta_path)

## 6. Creating Incremental Dataset

In [0]:
incremental_data = [
    ("CA-2016-152156", "Technology", 1200.0, 4),
    ("CA-2016-138688", "Furniture", 850.0, 2),
    ("CA-2025-999999", "Office Supplies", 300.0, 5)
]

inc_df = spark.createDataFrame(
    incremental_data,
    ["Order ID", "Category", "Sales", "Quantity"]
)

In [0]:
inc_df.show()

+--------------+---------------+------+--------+
|      Order ID|       Category| Sales|Quantity|
+--------------+---------------+------+--------+
|CA-2016-152156|     Technology|1200.0|       4|
|CA-2016-138688|      Furniture| 850.0|       2|
|CA-2025-999999|Office Supplies| 300.0|       5|
+--------------+---------------+------+--------+



## 7. Prepare Incremental Data

In [0]:
inc_df = inc_df.select([
    col(c).alias(c.replace(" ", "_"))
    for c in inc_df.columns
])
display(inc_df)

Order_ID,Category,Sales,Quantity
CA-2016-152156,Technology,1200.0,4
CA-2016-138688,Furniture,850.0,2
CA-2025-999999,Office Supplies,300.0,5


## 8. Merge Data

In [0]:
delta_table.alias("target") \
.merge(
    inc_df.alias("source"),
    "target.Order_ID = source.Order_ID"
) \
.whenMatchedUpdate(
    set={
        "Category": "source.Category",
        "Sales": "source.Sales",
        "Quantity": "source.Quantity"
    }
) \
.whenNotMatchedInsert(
    values={
        "Order_ID": "source.Order_ID",
        "Category": "source.Category",
        "Sales": "source.Sales",
        "Quantity": "source.Quantity"
    }
) \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## 9. Validate Results

In [0]:
delta_table.toDF().show(10)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|       State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name| Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+------------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|     5|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|   Consumer|United States|Fort Lauderdale|     Florida|      33311|  South|OFF-ST-10000760|Office Supplies| 

In [0]:
print("Total Rows :", delta_table.toDF().count())


Total Rows : 9995


## 10. Final Output

In [0]:
delta_table.toDF().show(20)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|     Customer_Name|    Segment|      Country|           City|         State|Postal_Code| Region|     Product_ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+---------+
|     5|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|   Consumer|United States|Fort Lauderdale|       Florida|      33311|  South|OFF-ST-10000760|

## Summary

- Loaded the Superstore dataset from Unity Catalog.
- Renamed columns for Delta compatibility.
- Created a Delta table.
- Cleaned duplicate and missing values.
- Created an incremental dataset.
- Performed a Delta Lake MERGE.
- Validated the merge results.
- Displayed the final dataset.